## Street Slope Calculation via DTM
In this section, we use the 2-meter resolution Digital Terrain Model (DTM) from ICGC to extract the elevation at the start and end of every street segment. This allows us to calculate the percentage slope, which is a critical factor for vehicle acoustic emissions.

In [ ]:
import geopandas as gpd
import rasterio
import numpy as np

# Load the street network geometry layer
noise_streets = gpd.read_file("../layers/BCN_noise_streets.gpkg")

print("Libraries and spatial data loaded. Ready for the DTM!")

Libraries loaded. Ready for the DTM!


In [2]:
# Define the path to your TIF file. 
# Remember: the .tfw file MUST be in the exact same folder and have the exact same name!
dtm_path = "../layers/Digital_terrain_model/elevacions-terreny-lidar-Catalunya-2m-2008-2011tif1777940893149.tif"

### 3.1 Extracting Elevations (Z) from the Raster
We open the DTM and "sample" the pixels that correspond to the coordinates of the first and last node of every street (LineString). We use a `try-except` block to prevent the code from crashing if a street goes outside the boundaries of our raster file.

In [3]:
z_start_list = []
z_end_list = []

print("Sampling elevations (this might take a few seconds)...")

with rasterio.open(dtm_path) as src:
    for geom in noise_streets.geometry:
        if geom.geom_type == 'LineString':
            # Extract X and Y of the first and last point of the line
            start_coord = geom.coords[0]
            end_coord = geom.coords[-1]
            
            try:
                # Extract the Z value from the raster for those exact coordinates
                z_start = next(src.sample([(start_coord[0], start_coord[1])]))[0]
                z_end = next(src.sample([(end_coord[0], end_coord[1])]))[0]
            except Exception:
                # If the point falls outside the raster, assign NaN
                z_start, z_end = np.nan, np.nan
        else:
            z_start, z_end = np.nan, np.nan
            
        z_start_list.append(z_start)
        z_end_list.append(z_end)

noise_streets['elev_start_m'] = z_start_list
noise_streets['elev_end_m'] = z_end_list

print("Elevations extracted and added to the dataframe!")

Sampling elevations (this might take a few seconds)...


NameError: name 'noise_streets' is not defined